# Planejamento de rotas hospitalares

Este notebook executa o fluxo completo sem exigir um terminal. As etapas estão separadas para permitir a inspeção dos dados, a alteração dos parâmetros e a análise dos resultados. No VS Code, selecione o kernel da `.venv` e use **Executar tudo**.

## 1. Preparação do ambiente

A célula identifica a raiz do repositório e instala o projeto no próprio kernel. Ela pode ser executada novamente sem recriar o ambiente.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT), '--quiet'])
sys.path.insert(0, str(ROOT / 'src'))
print(f'Projeto preparado em: {ROOT}')

## 2. Importações

Os módulos são separados por responsabilidade: dados, otimização, comparação, visualização e relatórios.

In [ ]:
import random
from IPython.display import IFrame, Markdown, clear_output, display
import matplotlib.pyplot as plt

from hospital_routes.baselines import nearest_neighbor
from hospital_routes.genetic import GAConfig, GeneticOptimizer
from hospital_routes.io import load_problem, save_solution
from hospital_routes.models import Delivery, Depot, Problem, Vehicle
from hospital_routes.reporting import GeminiReportGenerator
from hospital_routes.visualization import save_convergence_plot, save_route_map

## 3. Configuração do experimento

Altere os valores abaixo e execute novamente a partir desta célula. `USAR_CENARIO_PADRAO = True` lê o arquivo do projeto. Com `False`, um novo cenário reproduzível é criado a partir da semente. O elitismo aceita zero.

In [ ]:
USAR_CENARIO_PADRAO = True
NUMERO_ENTREGAS = 10
NUMERO_VEICULOS = 3
CAPACIDADE_KG = 55.0
AUTONOMIA_KM = 65.0

TAMANHO_POPULACAO = 120
GERACOES = 300
TAXA_CROSSOVER = 0.90
TAXA_MUTACAO = 0.20
ELITISMO = 4
SEMENTE = 42

## 4. Construção do cenário

Cada ponto representa uma entrega. O hospital é o depósito de saída e retorno e não entra nessa contagem. Coordenadas, demandas e prioridades podem ser alteradas diretamente no objeto `problem` após esta célula.

In [ ]:
if USAR_CENARIO_PADRAO:
    problem = load_problem(ROOT / 'data' / 'deliveries.json')
else:
    rng = random.Random(SEMENTE)
    depot = Depot('Hospital Central', rng.uniform(-13.005, -12.94), rng.uniform(-38.53, -38.45))
    deliveries = tuple(
        Delivery(
            id=f'E{i + 1:02d}', name=f'Ponto {i + 1:02d}',
            latitude=rng.uniform(-13.05, -12.89), longitude=rng.uniform(-38.57, -38.33),
            demand_kg=round(rng.uniform(4, CAPACIDADE_KG * 0.30), 1),
            priority=rng.randint(1, 3), service_minutes=10,
        ) for i in range(NUMERO_ENTREGAS)
    )
    vehicles = tuple(Vehicle(f'VEIC-{i + 1:02d}', CAPACIDADE_KG, AUTONOMIA_KM) for i in range(NUMERO_VEICULOS))
    problem = Problem(depot, deliveries, vehicles)

print(f'Hospital: {problem.depot.latitude:.6f}, {problem.depot.longitude:.6f}')
print(f'Entregas: {len(problem.deliveries)} | Veículos: {len(problem.vehicles)}')
for delivery in problem.deliveries:
    print(f'{delivery.id}: ({delivery.latitude:.6f}, {delivery.longitude:.6f}) | {delivery.demand_kg:.1f} kg | prioridade {delivery.priority}')

## 5. Configuração do algoritmo genético

A representação é uma permutação das entregas. A função de aptidão combina distância, atendimento prioritário e penalidades por excesso de carga ou autonomia.

In [ ]:
config = GAConfig(
    population_size=TAMANHO_POPULACAO, generations=GERACOES,
    crossover_rate=TAXA_CROSSOVER, mutation_rate=TAXA_MUTACAO,
    elite_size=ELITISMO, seed=SEMENTE,
)
optimizer = GeneticOptimizer(problem, config)
baseline = nearest_neighbor(optimizer)
print(f'Fitness do vizinho mais próximo: {baseline.fitness:.2f}')

## 6. Evolução e acompanhamento da convergência

Os gráficos possuem escalas independentes. O primeiro acompanha o melhor indivíduo; o segundo acompanha a média da população. A saída é atualizada periodicamente durante a evolução.

In [ ]:
solution = None
for stats, solution in optimizer.evolve():
    if stats.generation % 10 == 0 or stats.generation == GERACOES - 1:
        clear_output(wait=True)
        generations = [item.generation for item in optimizer.history]
        fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
        axes[0].plot(generations, [item.best for item in optimizer.history], color='#24966f')
        axes[0].set_title('Melhor fitness'); axes[0].set_ylabel('Fitness'); axes[0].grid(alpha=.25)
        axes[1].plot(generations, [item.mean for item in optimizer.history], color='#657f9f')
        axes[1].set_title('Fitness médio'); axes[1].set_xlabel('Geração'); axes[1].set_ylabel('Fitness'); axes[1].grid(alpha=.25)
        fig.tight_layout(); display(fig); plt.close(fig)
        print(f'Geração {stats.generation} | melhor: {stats.best:.2f} | média: {stats.mean:.2f}')

assert solution is not None
print(f'Concluído em {len(optimizer.history)} gerações.')

## 7. Análise da solução

A comparação usa a mesma função de aptidão. A viabilidade exige que todas as rotas respeitem capacidade e autonomia.

In [ ]:
improvement = 100 * (baseline.fitness - solution.fitness) / baseline.fitness
print(f'Fitness final: {solution.fitness:.2f}')
print(f'Distância total: {solution.total_distance_km:.2f} km')
print(f'Solução viável: {solution.feasible}')
print(f'Redução frente ao vizinho mais próximo: {improvement:.2f}%')
for i, route in enumerate(solution.routes):
    metric = solution.metrics[i]
    stops = ' → '.join(problem.deliveries[j].id for j in route) or 'não utilizado'
    print(f'{problem.vehicles[i].id}: {stops} | {metric.distance_km:.2f} km | {metric.load_kg:.1f} kg')

## 8. Geração dos entregáveis

Esta etapa salva o JSON, o mapa HTML, o gráfico final e o relatório operacional na pasta `outputs`.

In [ ]:
OUTPUT = ROOT / 'outputs'
OUTPUT.mkdir(exist_ok=True)
save_solution(OUTPUT / 'solution.json', problem, solution)
save_route_map(problem, solution, OUTPUT / 'routes_map.html')
save_convergence_plot(optimizer.history, OUTPUT / 'convergence.png')
report = GeminiReportGenerator().generate(problem, solution)
(OUTPUT / 'daily_report.md').write_text(report, encoding='utf-8')
print('Arquivos gerados:')
for name in ('solution.json', 'routes_map.html', 'convergence.png', 'daily_report.md'):
    print(' -', OUTPUT / name)

## 9. Mapa e relatório no notebook

O mapa interativo e as instruções operacionais são apresentados abaixo sem abrir outro programa.

In [ ]:
display(IFrame(src=str(OUTPUT / 'routes_map.html'), width='100%', height=550))
display(Markdown(report))

## 10. Integração com a LLM

O relatório acima foi produzido pelo modelo Gemini configurado de forma centralizada no projeto. Para testes determinísticos sem rede, ainda é possível utilizar `LocalReportGenerator`.